# **Aspect-Based Sentiment Analysis (ABSA) — Simple Inference**

## Imports

In [2]:
!pip -q install transformers accelerate torch
!huggingface-cli login

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `colab-gemma` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /roo

In [3]:
import re, torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, pipeline
import pandas as pd
import numpy as np
from tqdm import tqdm

## Load Data

In [4]:
test_data = pd.read_csv('https://drive.google.com/uc?id=1_quwLStprNMufdr-d8FagQUPbScSLdJy')
fewshot_data = pd.read_csv('https://drive.google.com/uc?id=1veKHD5Nl52ODG265OcUDFLsby_RcUmXv')

## Load Model

### Decoder-Only

In [5]:
MODEL_NAME = "google/gemma-3-1b-it"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

if tok.pad_token_id is None:
    tok.pad_token_id = tok.eos_token_id
tok.padding_side = "left"
model.generation_config.pad_token_id = tok.pad_token_id
model.generation_config.eos_token_id = tok.eos_token_id

gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tok,
    device=0 if torch.cuda.is_available() else -1
)

IS_DECODER_ONLY = True

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Device set to use cuda:0


## Prompt + Input

In [6]:
def build_fewshot_prompt(df: pd.DataFrame, n_examples_per_aspect: int = 3, max_total_examples: int = 50) -> str:
    unique_aspects = df["aspect"].unique()
    examples = []

    for aspect in unique_aspects:
        subset = df[df["aspect"] == aspect]
        n_pick = min(n_examples_per_aspect, len(subset))
        sampled = subset.sample(n=n_pick, random_state=42)
        for _, row in sampled.iterrows():
            example = (
                f'Sentence: "{row["sentence"].strip()}"\n'
                f'Aspect: "{row["aspect"].strip()}"\n'
                f'Answer: {row["sentiment"].strip().lower()}\n'
            )
            examples.append(example)

    # Shuffle to avoid bias and limit total examples
    np.random.seed(42)
    np.random.shuffle(examples)
    examples = examples[:max_total_examples]

    joined_examples = "\n".join(examples)

    PROMPT_FEW = (
        "You are an ABSA assistant. Decide the sentiment towards the given aspect.\n"
        "Answer with ONE WORD only from: positive, negative, neutral, conflict.\n\n"
        "Examples:\n"
        f"{joined_examples}\n"
        "Now classify the following pair."
    )

    return PROMPT_FEW

In [7]:
def build_cot_with_examples(fewshot_df, n_examples: int = 1, random_state: int = 42):
    examples = []
    sampled = fewshot_df.sample(n=min(n_examples, len(fewshot_df)), random_state=random_state)
    for _, r in sampled.iterrows():
        ex = (
            f'Sentence: "{r["sentence"].strip()}"\n'
            f'Aspect: "{r["aspect"].strip()}"\n'
            f"Step 1: Opinion phrase → (extract)\n"
            f"Step 2: Polarity toward {r['aspect'].strip()} → {r['sentiment'].strip().lower()}\n"
            f"Step 3: Final answer (one word only): {r['sentiment'].strip().lower()}\n\n"
        )
        examples.append(ex)

    examples_block = "\n".join(examples)
    prompt = (
        "You are an ABSA assistant. Think step by step before deciding the label.\n"
        "Steps:\n"
        "1) Extract opinion phrases related to the aspect.\n"
        "2) Determine polarity towards that aspect only.\n"
        "3) Output the final label as ONE WORD from: positive, negative, neutral, conflict.\n\n"
        "Examples:\n"
        f"{examples_block}"
        "Now analyze the following.\n"
        "Final answer (one word only):"
    )
    return prompt

In [15]:
# === Prompt Variants ===

PROMPT_ZERO = (
    "Classify the sentiment towards the given aspect in the sentence below.\n"
    "Answer with ONE WORD only from: positive, negative, neutral, conflict."
)

PROMPT_SIMPLE = (
    "Classify the sentiment towards the aspect as one of: positive, negative, neutral, conflict.\n"
)


PROMPT_FEW = build_fewshot_prompt(fewshot_data, n_examples_per_aspect=1, max_total_examples=2)

PROMPT_CoT = (
    "You are an ABSA assistant. Think step by step before deciding the label.\n"
    "Steps:\n"
    "1) Extract opinion phrases related to the aspect.\n"
    "2) Determine polarity towards that aspect only.\n"
    "3) Output the final label as ONE WORD from: positive, negative, neutral, conflict.\n"
    "Final answer (one word only):"
)

PROMPT_KNOWLEDGE = (
    "You are an Aspect-Based Sentiment Analysis (ABSA) assistant.\n"
    "Follow these steps carefully:\n"
    "1) Recall what people usually think about the given aspect in general.\n"
    "2) Based on the specific sentence, identify the opinion phrases related to that aspect.\n"
    "3) Determine the sentiment polarity toward that aspect only.\n"
    "4) Finally, output ONE WORD ONLY from: positive, negative, neutral, conflict.\n\n"
    "Think step by step, but end your answer with only the final label."
)

PROMPT_FEW_CoT = build_cot_with_examples(fewshot_data, n_examples=1)

In [9]:
LABELS = ["positive", "negative", "neutral", "conflict"]

## Inference + Post-Processing

In [10]:
def generate_text(prompt: str, max_new_tokens: int = 16, do_sample: bool = False) -> str:
    if IS_DECODER_ONLY:
        # decoder-only → pastikan tidak mengirim flag sampling saat do_sample=False
        return gen(prompt, max_new_tokens=max_new_tokens, do_sample=do_sample, return_full_text=False)[0]["generated_text"]
    else:
        # seq2seq → tidak pakai return_full_text
        return gen(prompt, max_new_tokens=max_new_tokens, do_sample=do_sample)[0]["generated_text"]

In [11]:
CANON = {
    "positive":"positive","pos":"positive","positif":"positive","+":"positive",
    "negative":"negative","neg":"negative","negatif":"negative","-":"negative",
    "neutral":"neutral","neu":"neutral","netral":"neutral","none":"neutral",
    "conflict":"conflict","mixed":"conflict","both":"conflict","conflicted":"conflict"
}
PAT = re.compile(r"\b(positive|negative|neutral|conflict|pos|neg|neu|netral|positif|negatif|mixed|both|conflicted)\b", re.I)

def normalize_label(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return "neutral"
    m = PAT.findall(text.strip().lower())
    if m:
        return CANON.get(m[-1].lower(), "neutral")
    return "neutral"

### Zero-shot

In [ ]:
# === Inference loop ===
prompts = [
    f"{PROMPT_ZERO}\n\nSentence: \"{row['sentence']}\"\nAspect: \"{row['aspect']}\"\nAnswer (one of: positive, negative, neutral, conflict):\n"
    for _, row in test_data.iterrows()
]

batch_size = 8
predictions = []

for i in tqdm(range(0, len(prompts), batch_size)):
    batch_prompts = prompts[i:i+batch_size]
    outputs = gen(batch_prompts, max_new_tokens=16, do_sample=False, return_full_text=False)
    for batch_out in outputs:
        text = batch_out[0]["generated_text"]
        predictions.append(normalize_label(text))

# === Save to CSV ===
output = pd.DataFrame({
    "id": test_data["id"],
    "sentiment": predictions
})

output.to_csv("prediction_zero_2.csv", index=False)

100%|██████████| 125/125 [13:00<00:00,  6.25s/it]


## Simple

In [16]:
# === Inference loop ===
prompts = [
    f"{PROMPT_SIMPLE}\n\nSentence: \"{row['sentence']}\"\nAspect: \"{row['aspect']}\"\nAnswer (one of: positive, negative, neutral, conflict):\n"
    for _, row in test_data.iterrows()
]

batch_size = 8
predictions = []

for i in tqdm(range(0, len(prompts), batch_size)):
    batch_prompts = prompts[i:i+batch_size]
    outputs = gen(batch_prompts, max_new_tokens=16, do_sample=False, return_full_text=False)
    for batch_out in outputs:
        text = batch_out[0]["generated_text"]
        predictions.append(normalize_label(text))

# === Save to CSV ===
output = pd.DataFrame({
    "id": test_data["id"],
    "sentiment": predictions
})

output.to_csv("prediction_simple_1.csv", index=False)

100%|██████████| 125/125 [12:38<00:00,  6.07s/it]


### Few-shot

In [ ]:
# === Inference loop ===
prompts = [
    f"{PROMPT_FEW}\n\nSentence: \"{row['sentence']}\"\nAspect: \"{row['aspect']}\"\nAnswer (one of: positive, negative, neutral, conflict):\n"
    for _, row in test_data.iterrows()
]

batch_size = 8
predictions = []

for i in tqdm(range(0, len(prompts), batch_size)):
    batch_prompts = prompts[i:i+batch_size]
    outputs = gen(batch_prompts, max_new_tokens=16, do_sample=False, return_full_text=False)
    for batch_out in outputs:
        text = batch_out[0]["generated_text"]
        predictions.append(normalize_label(text))

# === Save to CSV ===
output = pd.DataFrame({
    "id": test_data["id"],
    "sentiment": predictions
})

output.to_csv("prediction_few_5.csv", index=False)

100%|██████████| 125/125 [13:13<00:00,  6.35s/it]


### Chain of Thoughts

In [ ]:
# === Inference loop ===
prompts = [
    f"{PROMPT_CoT}\n\nSentence: \"{row['sentence']}\"\nAspect: \"{row['aspect']}\"\nAnswer (one of: positive, negative, neutral, conflict):\n"
    for _, row in test_data.iterrows()
]

batch_size = 8
predictions = []

for i in tqdm(range(0, len(prompts), batch_size)):
    batch_prompts = prompts[i:i+batch_size]
    outputs = gen(batch_prompts, max_new_tokens=16, do_sample=False, return_full_text=False)
    for batch_out in outputs:
        text = batch_out[0]["generated_text"]
        predictions.append(normalize_label(text))

# === Save to CSV ===
output = pd.DataFrame({
    "id": test_data["id"],
    "sentiment": predictions
})

output.to_csv("prediction_cot_4.csv", index=False)

100%|██████████| 125/125 [12:43<00:00,  6.11s/it]


### Chain of Thoughts + Self Consistency

In [ ]:
def generate_consistent_label(prompt: str, n_samples: int = 5, max_new_tokens: int = 32) -> str:
    preds = []
    for _ in range(n_samples):
        out = gen(prompt, max_new_tokens=max_new_tokens, temperature=0.8, top_p=0.9, return_full_text=False)[0]
        preds.append(normalize_label(out["generated_text"]))
    # Majority vote
    if preds:
        return max(set(preds), key=preds.count)
    return "neutral"

# === Inference Loop ===
predictions = []
for _, row in tqdm(test_data.iterrows(), total=len(test_data)):
    prompt = (
        f"{PROMPT_CoT}\n\n"
        f"Sentence: \"{row['sentence']}\"\n"
        f"Aspect: \"{row['aspect']}\"\n"
        f"Answer (one of: positive, negative, neutral, conflict):\n"
    )
    label = generate_consistent_label(prompt, n_samples=5)
    predictions.append(label)

# === Save to CSV ===
output = pd.DataFrame({
    "id": test_data["id"],
    "sentiment": predictions
})

output.to_csv("prediction_selfconsistency_2.csv", index=False)

100%|██████████| 1000/1000 [2:06:07<00:00,  7.57s/it]


### Knowledge Augmentation

In [ ]:
# === Inference loop ===
prompts = [
    f"{PROMPT_KNOWLEDGE}\n\nSentence: \"{row['sentence']}\"\nAspect: \"{row['aspect']}\"\nAnswer (one of: positive, negative, neutral, conflict):\n"
    for _, row in test_data.iterrows()
]

batch_size = 8
predictions = []

for i in tqdm(range(0, len(prompts), batch_size)):
    batch_prompts = prompts[i:i+batch_size]
    outputs = gen(batch_prompts, max_new_tokens=16, do_sample=False, return_full_text=False)
    for batch_out in outputs:
        text = batch_out[0]["generated_text"]
        predictions.append(normalize_label(text))

# === Save to CSV ===
output = pd.DataFrame({
    "id": test_data["id"],
    "sentiment": predictions
})

output.to_csv("prediction_ka_2.csv", index=False)

100%|██████████| 125/125 [13:14<00:00,  6.36s/it]


### Chain of Thoughts + Few-shot

In [ ]:
# === Inference loop ===
prompts = [
    f"{PROMPT_FEW_CoT}\n\nSentence: \"{row['sentence']}\"\nAspect: \"{row['aspect']}\"\nAnswer (one of: positive, negative, neutral, conflict):\n"
    for _, row in test_data.iterrows()
]

batch_size = 8
predictions = []

for i in tqdm(range(0, len(prompts), batch_size)):
    batch_prompts = prompts[i:i+batch_size]
    outputs = gen(batch_prompts, max_new_tokens=16, do_sample=False, return_full_text=False)
    for batch_out in outputs:
        text = batch_out[0]["generated_text"]
        predictions.append(normalize_label(text))

# === Save to CSV ===
output = pd.DataFrame({
    "id": test_data["id"],
    "sentiment": predictions
})

output.to_csv("prediction_cot_few_3.csv", index=False)

100%|██████████| 125/125 [12:59<00:00,  6.24s/it]
